# Data Exploration
## Data loading and exploration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Loading the insurance dataset
insurance_data_path = 'insurance.csv'
insurance = pd.read_csv(insurance_data_path)

insurance.head()

### 1.1 Understand the dataset
#### 1.1.1 Check the dataset dimensions

In [ ]:
insurance.shape

**Observation:**
- Total rows: 1338 (dataset size)
- Total columns: 7 (6 features + 1 target)

#### 1.1.2 Check the column information
Pay attention to:
* Number of entries.
* Data types (int64, float64, object).
* Non-null counts.
* Whether there are missing values.

In [ ]:
insurance.info()

**Observation:**

| Item | Result | Observation |
|---|---|---|
| Age | 1,272 non-null | 66 missing |
| Sex | 1,272 non-null | 66 missing |
| BMI | 1,272 non-null | 66 missing |
| Children | 1,272 non-null | 66 missing |
| Smoker | 1,272 non-null | 66 missing |
| Region | 1,272 non-null | 66 missing |
| Charges | 1,284 non-null | 54 missing |

> Notice that the first six columns have exactly the same number of non-null values. This might indicate that some records have multiple missing fields.

#### 1.1.3 Generate descriptive statistics
This gives us:
* Mean and standard deviation.
* Minimum and maximum values.
* 25th, 50th and 75th percentiles.

In [ ]:
insurance.describe()

#### 1.1.4 Check categorical values

In [ ]:
insurance.describe(include='object')

### Issues discovered
#### Issue 1: Missing values – Needs investigation
There are 66 missing values in each of the 6 feature columns and 54 missing values in `charges`.

#### Issue 2: Invalid age – Anomaly
The minimum age is -64. A person's age cannot be negative.

#### Issue 3: Invalid children – Anomaly
The minimum children value is -4. The number of dependents cannot be negative.

#### Issue 4: Categorical inconsistencies – Needs investigation
* `sex` has 6 unique values, although only male and female are expected.
* `region` has 8 unique values, although only 4 regions are expected.

#### Issue 5: Incorrect data type – Needs conversion
`charges` is currently an object instead of a numeric column.

The output also shows `$nan` as the most frequent value, suggesting that some charges may contain currency symbols or text representations of missing values.

### 2.1 Investigate missing values

In [ ]:
insurance.isnull().sum()

Then, calculate the missing percentage:

In [ ]:
(insurance.isnull().sum() / len(insurance)) * 100

### 2.2 Investigate invalid numerical values
#### Age

In [ ]:
insurance[insurance['age'] < 0]

#### Children

In [ ]:
insurance[insurance['children'] < 0]

#### BMI

In [ ]:
insurance['bmi'].describe()

### 2.3 Investigate categorical inconsistencies
#### Sex

In [ ]:
insurance['sex'].value_counts(dropna=False)

#### Smoker

In [ ]:
insurance['smoker'].value_counts(dropna=False)

#### Region

In [ ]:
insurance['region'].value_counts(dropna=False)

### Review: Data quality assessment
#### Missing values
All 6 feature columns have exactly 66 missing values. This suggests that some rows might have been corrupted or partially removed.
> **Suggestion:** Do not immediately remove every row containing a missing value. Investigate whether the missing values occur in the same records.

#### Invalid numerical values
* **Age**: Negative ages are impossible. These records need correction or removal.
* **Children**: Negative numbers of children are also impossible.
* **BMI**: No obvious invalid BMI values from the descriptive statistics.

One important distinction:
* Negative age is an invalid value.
* Negative children is an invalid value.
* High BMI is not automatically an error.

> **Suggestion:** Treat negative ages and children as invalid records rather than converting them into positive values. For example, changing -40 into 40 assumes that the original value was entered with an incorrect sign. We don't have evidence to justify that assumption.

#### Categorical inconsistencies
* **Sex**: standardize into lowercase `male` and `female`
* **Smoker**: standardize the capitalization
* **Region**: standardize the capitalization
> Convert categorical text to lowercase and remove unnecessary spaces before checking the unique values again.

### Recommended cleaning strategy
| Problem | Recommended treatment |
|---|---|
| Missing feature values | Investigate overlap, then consider imputation or row removal |
| Missing `charges` | Remove records without a target value from training |
| Negative `age` | Remove invalid records |
| Negative `children` | Remove invalid records |
| Sex inconsistencies | Standardize into `male` / `female` |
| Smoker capitalization | Convert to lowercase |
| Region capitalization | Convert to lowercase |
| Charges currency symbols | Remove symbols and convert to numeric |

We should also preserve the original dataset. This is useful when experimenting with different cleaning approaches.

In [ ]:
# Keep an untouched copy of the raw data
insurance_original = insurance.copy()

### 2.4 Investigate missing value overlap
This is to understand whether missing values are concentrated in the same rows. For example:
* If a row has all six features missing, removing it might be reasonable.
* If a row has only BMI missing but has valid charges, we might preserve it and impute BMI.

In [ ]:
insurance[insurance.isnull().any(axis=1)]

In [ ]:
insurance[insurance.isnull().all(axis=1)]

### 2.5 Investigate the charges column
This is to understand the currency symbol and `$nan` values before converting the column.

In [ ]:
insurance['charges'].head(20)

In [ ]:
insurance['charges'].unique()[:20]

### Interpretation of the findings
Only 2 rows have every column missing, at indices 447 and 780. Most other rows with missing values still contain useful information.

| Situation | Example | Possible treatment |
|---|---|---|
| Entire row is empty | Index 447 and 780 | Remove the rows |
| Some features are missing, but charges are available | Age or BMI is missing | Impute missing features or remove selected rows |
| Charges are missing | `$nan` or `null` | Convert these to missing values and exclude these rows from model training |

We cannot train a supervised regression model using rows without a known target (`charges`). The model needs actual charges to learn the relationship between customer characteristics and insurance costs.

However, we should not automatically remove every row with missing features. That could discard useful training data.

Based on the first 20 records and the unique values, we can identify the main inconsistencies in the `charges` column:
1. **Inconsistent currency symbol**: Some values contain a `$` symbol, while others contain only numeric characters. For example, `16884.924` and `$4449.462` represent the same type of information but are stored in different formats.
2. **Object data type**: The values are stored as strings rather than numbers. This prevents us from directly performing numerical calculations or using the column as a regression target.
3. **Potential missing values**: Previous investigations also found `$nan` values. These represent missing or invalid charge values and should be converted to actual `NaN` values.

> The `charges` column requires cleaning before modelling. We need to remove the `$` symbol and convert the values into a numeric data type. Invalid or missing values should become `NaN` so that we can handle them properly.